We applied the 3DSC methodology https://github.com/aimat-lab/3DSC to handle chemical formulas, including the definitions for exact matching, similarities, doping, and unmatched cases.

In [2]:
from pymatgen.core import Structure
from pymatgen.io.cif import CifWriter
from pymatgen.transformations.standard_transformations import OrderDisorderedStructureTransformation
order = OrderDisorderedStructureTransformation()
import numpy as np
from structure import get_chem_dict,get_formula_similarity,check_for_doping,get_doping_structure
def test_similarity(formula_sc, formula_2):
    chemdict_sc = get_chem_dict(formula_sc)
    chemdict_2 = get_chem_dict(formula_2)
    return  get_formula_similarity(chemdict_sc, chemdict_2)
test_similarity('Ba3Ca1Cu6.58Fe0.42La2.5Y0.5O16.599','Ba1.286Ca0.429Cu2.764Fe0.236La1.071Y0.214O7.24')

(2, np.float64(0.016915842183891423))

In [4]:
import os
fn='cif/'
names={}
for name in os.listdir(fn):
    if name.endswith('.cif'):
        structure=Structure.from_file(fn+name)
        reference_formula=structure.formula.replace(" ","")
        names[name]=reference_formula #cifs names

/home/hoanguyen/miniconda3/envs/doduy_py312/lib/python3.12/site-packages/pymatgen/core/structure.py:3112: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/hoanguyen/miniconda3/envs/doduy_py312/lib/python3.12/site-packages/pymatgen/core/structure.py:3112: UserWarning: Issues encountered while parsing CIF: 20 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


Match cifs with Chemical formulas in Supercon.

The order transformation method retains only ordered structures with the lowest Ewald energy, to generate ordered cifs.

In [6]:
from pymatgen.core import Structure
from pymatgen.transformations.standard_transformations import OrderDisorderedStructureTransformation
order = OrderDisorderedStructureTransformation()
import openpyxl
workbook = openpyxl.load_workbook('SuperBand.xlsx')	
sheet = workbook['Sheet1']  
for ii in range(10):
    print("ii=",ii)
    for i in range (2,len(sheet["A"])+1):
        if not sheet["B"+str(i)].value:
            continue
        tmp=True
        #print(sheet["B"+str(i)].value)
        for name in os.listdir(fn):
            try:
                index,_=test_similarity(sheet["B"+str(i)].value,names[name])
                #index=1:exact matching, 2:similarity, 3:doping, 4:unmatch
            except:
                continue
            if sheet["I"+str(i)].value =='found':
                tmp=False
                continue
            if index<2.1:
                sheet["H"+str(i)]=os.path.splitext(name)[0]
                tmp=False
                print(sheet["B"+str(i)].value,name,'found')
                sheet["I"+str(i)]='found'
                break
        if tmp:
            name=str(sheet["G"+str(i)].value)+'.cif'
            try:
                structure=Structure.from_file(fn+name)
            except:
                print(sheet["B"+str(i)].value,str(sheet["G"+str(i)].value),'nofound')
                sheet["I"+str(i)]='nofound'
                continue
            if check_for_doping(structure): 
                if len(structure)>40:#too many atoms to be doped
                    print(sheet["B"+str(i)].value,name,'huge')
                    sheet["H"+str(i)]=os.path.splitext(name)[0]
                    sheet["I"+str(i)]='huge'
                    continue
                try:
                    structure=order.apply_transformation(structure,return_ranked_list=100)[0]["structure"]
                except:
                    sheet["H"+str(i)]=os.path.splitext(name)[0]
                    print(sheet["B"+str(i)].value,str(sheet["A"+str(i)].value),'error')
                    sheet["I"+str(i)]='error'
                    continue
            cif=get_doping_structure(sheet["B"+str(i)].value,structure)
            try:
                index,_=test_similarity(sheet["B"+str(i)].value,cif.formula.replace(" ",""))
            except:
                index=4
            if index<2.1:
                sheet["H"+str(i)]=sheet["A"+str(i)].value
                print(sheet["B"+str(i)].value,str(sheet["A"+str(i)].value),'doped')
                sheet["I"+str(i)]='doped'
            else:
                sheet["H"+str(i)]=sheet["A"+str(i)].value
                print(sheet["B"+str(i)].value,str(sheet["A"+str(i)].value),'next')
                sheet["I"+str(i)]='next'
            output_cif = CifWriter(cif)
            output_cif.write_file(fn+str(sheet["A"+str(i)].value)+'.cif')
            names[str(sheet["A"+str(i)].value)+'.cif']=cif.formula.replace(" ","")

ii= 0
Ag0.02Ge2Pd1.98Sr1 14118 nofound
Ag0.15Sn0.85Te1 15674 nofound
Ag0.1Ge2Pd1.9Sr1 14118 nofound
Ag0.1In0.9Te1 14351 nofound
Ag0.2Ba1Si1.8 15855 nofound
Ag0.2Ge2Pd1.8Sr1 14118 nofound
Ag0.438Hg0.562 10007 doped
Ag0.25Sn0.75Te1 10008.cif found
Ag0.55Hg0.45 10009.cif found
Ag0.5Ba1Si1.5 10010.cif found
Ag0.5Pd0.5Th2 10011.cif found
Ag1Sn1Te2 10012.cif found
Ag0.625Al0.375 10013 doped
Ag0.65Al0.35 10015.cif found
Ag2Al1 10015.cif found
Ag0.6Al0.4 10013.cif found
Ag0.76Se2Sn1.24 10017 doped
Ag0.7Al0.3 10018 doped
Ag0.7Hg0.3 10019 doped
Ag0.7Zn0.3 10020 doped
Ag0.81In0.19 10051 nofound
Ag0.81Se2Sn1.19 10017.cif found
Ag0.85Se2Sn1.15 10017.cif found
Ag0.8Ga0.2 10024 doped
Ag0.92Se2Sn1.08 10038.cif found
Ag0.95Se2Sn1.05 10038.cif found
Ag0.9S0.2Se1.8Sn1.1 10027 next
Ag1B2 10028.cif found
Ag1Be2 10029.cif found
Ag1Ce1Sb2 10030.cif found
Ag1Cl2 10031 next
Ag1In2 10032.cif found
Ag1La1 10033.cif found
Ag1La1Sb2 10034.cif found
Ag1Mg1 10035.cif found
Ag1Mo6S8 10036.cif found
Ag1S0.2Se1.8Sn1 10

/tmp/ipykernel_60138/4144120667.py:63: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ag2', 'Ag3', 'Ag4', 'Ag3', 'Cl0', 'Cl4', 'Cl5', 'Cl6', 'Cl7', 'Cl8']`.
  output_cif = CifWriter(cif)


Ag7N1O11 10060 nofound
Al0.01B0.99Li2Pt3 11036 nofound
Al0.01B2Mg0.99 15774 nofound
Al0.022B2Mg0.978 15774 nofound
Al0.02B2Mg0.98 15774 nofound
Al0.02Nb3Sn0.98 15051 nofound
Al0.02Si0.98V3 15643 nofound
Al0.03B2Mg0.97 15774 nofound
Al0.044B2Mg0.956 15774 nofound
Al0.04Nb3Sn0.96 15051 nofound
Al0.05B0.95Li2Pd3 11035 nofound
Al0.05B0.95Li2Pt3 11036 nofound
Al0.05B2Mg0.95 15774 nofound
Al0.05Nb3Sn0.95 15051 nofound
Al0.06B2Mg0.94 15774 nofound
Al0.06Nb3Sn0.94 15051 nofound
Al0.07B2Mg0.93 15774 nofound
Al0.085B2Mg0.915 15774 nofound
Al0.08B2Mg0.92 15774 nofound
Al0.08Nb3Sn0.92 15051 nofound
Al0.09B2Mg0.91 15774 nofound
Al0.09Nb3Sn0.91 15051 nofound
Al0.14B2Mg0.86 15774 nofound
Al0.15B2Mg0.85 15774 nofound
Al0.15Ga0.85Nb3 15793 nofound
Al0.15Ge0.85Nb3 14071 nofound
Al0.185B2Mg0.815 15774 nofound
Al0.188V0.812 10101 nofound
Al0.18La0.82 10239 nofound
Al0.19B2Mg0.81 15774 nofound
Al0.19Nb3Sn0.81 15051 nofound
Al0.1B0.9Li2Pd3 11035 nofound
Al0.1B2Mg0.9 15774 nofound
Al0.1Ca1Ga0.9Si1 12748 nofo

/tmp/ipykernel_60138/4144120667.py:63: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ag2', 'Ag3', 'Cl2', 'Ag3', 'Cl0', 'Cl4', 'Cl5', 'Cl6', 'Cl7', 'Cl8']`.
  output_cif = CifWriter(cif)


Al0.01B2Mg0.99 15774 nofound
Al0.022B2Mg0.978 15774 nofound
Al0.02B2Mg0.98 15774 nofound
Al0.02Nb3Sn0.98 15051 nofound
Al0.02Si0.98V3 15643 nofound
Al0.03B2Mg0.97 15774 nofound
Al0.044B2Mg0.956 15774 nofound
Al0.04Nb3Sn0.96 15051 nofound
Al0.05B0.95Li2Pd3 11035 nofound
Al0.05B0.95Li2Pt3 11036 nofound
Al0.05B2Mg0.95 15774 nofound
Al0.05Nb3Sn0.95 15051 nofound
Al0.06B2Mg0.94 15774 nofound
Al0.06Nb3Sn0.94 15051 nofound
Al0.07B2Mg0.93 15774 nofound
Al0.085B2Mg0.915 15774 nofound
Al0.08B2Mg0.92 15774 nofound
Al0.08Nb3Sn0.92 15051 nofound
Al0.09B2Mg0.91 15774 nofound
Al0.09Nb3Sn0.91 15051 nofound
Al0.14B2Mg0.86 15774 nofound
Al0.15B2Mg0.85 15774 nofound
Al0.15Ga0.85Nb3 15793 nofound
Al0.15Ge0.85Nb3 14071 nofound
Al0.185B2Mg0.815 15774 nofound
Al0.188V0.812 10101 nofound
Al0.18La0.82 10239 nofound
Al0.19B2Mg0.81 15774 nofound
Al0.19Nb3Sn0.81 15051 nofound
Al0.1B0.9Li2Pd3 11035 nofound
Al0.1B2Mg0.9 15774 nofound
Al0.1Ca1Ga0.9Si1 12748 nofound
Al0.1Ga0.9V3 15776 nofound
Al0.1Nb3Sn0.9 15051 nofo

/tmp/ipykernel_60138/4144120667.py:63: UserWarning: Site labels are not unique, which is not compliant with the CIF spec (https://www.iucr.org/__data/iucr/cifdic_html/1/cif_core.dic/Iatom_site_label.html):`['Ag2', 'Ag3', 'Ag3', 'Cl2', 'Cl0', 'Cl4', 'Cl5', 'Cl6', 'Cl7', 'Ag9']`.
  output_cif = CifWriter(cif)


Al0.05B2Mg0.95 15774 nofound
Al0.05Nb3Sn0.95 15051 nofound
Al0.06B2Mg0.94 15774 nofound
Al0.06Nb3Sn0.94 15051 nofound
Al0.07B2Mg0.93 15774 nofound
Al0.085B2Mg0.915 15774 nofound
Al0.08B2Mg0.92 15774 nofound
Al0.08Nb3Sn0.92 15051 nofound
Al0.09B2Mg0.91 15774 nofound
Al0.09Nb3Sn0.91 15051 nofound
Al0.14B2Mg0.86 15774 nofound
Al0.15B2Mg0.85 15774 nofound
Al0.15Ga0.85Nb3 15793 nofound
Al0.15Ge0.85Nb3 14071 nofound
Al0.185B2Mg0.815 15774 nofound
Al0.188V0.812 10101 nofound
Al0.18La0.82 10239 nofound
Al0.19B2Mg0.81 15774 nofound
Al0.19Nb3Sn0.81 15051 nofound
Al0.1B0.9Li2Pd3 11035 nofound
Al0.1B2Mg0.9 15774 nofound
Al0.1Ca1Ga0.9Si1 12748 nofound
Al0.1Ga0.9V3 15776 nofound
Al0.1Nb3Sn0.9 15051 nofound
Al0.215Nb0.785 15775 nofound
Al0.21B2Mg0.79 15774 nofound
Al0.22La0.78 10239 nofound
Al0.24Si0.76V3 15643 nofound
Al0.25Ti0.525V0.255 10100 nofound
Al1V3 10101 nofound
Al0.27La0.73 10239 nofound
Al0.28Mg0.18 10103 nofound
Al0.29B2Mg0.71 10126 nofound
Al0.2B2Mg0.8 10105 nofound
Al0.2B5Mo1.8 10106 n